# **SETUP**

In [1]:
import pathlib
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

PROJECT_ROOT = pathlib.Path().absolute().parent

train = pd.read_parquet(PROJECT_ROOT / "data" / "train.parquet")
ohe_oof = pd.read_parquet(PROJECT_ROOT / "data" / "features" / "002-one-hot-categoricals" / "oof.parquet")
std_oof = pd.read_parquet(PROJECT_ROOT / "data" / "features" / "003-standard-scale-numerics" / "oof.parquet")
kbins_oof = pd.read_parquet(PROJECT_ROOT / "data" / "features" / "005-kbins-discretize-numerics" / "oof.parquet")

test = pd.read_parquet(PROJECT_ROOT / "data" / "test.parquet")
ohe_test = pd.read_parquet(PROJECT_ROOT / "data" / "features" / "002-one-hot-categoricals" / "test.parquet")
std_test = pd.read_parquet(PROJECT_ROOT / "data" / "features" / "003-standard-scale-numerics" / "test.parquet")
kbins_test = pd.read_parquet(PROJECT_ROOT / "data" / "features" / "005-kbins-discretize-numerics" / "test.parquet")

X_train = (
    train
    .reset_index()[["id"]]
    .merge(ohe_oof, how="left", on="id")
    .merge(std_oof, how="left", on="id")
    .merge(kbins_oof, how="left", on="id")
)
X_test = (
    test
    .reset_index()[["id"]]
    .merge(ohe_test, how="left", on="id")
    .merge(std_test, how="left", on="id")
    .merge(kbins_test, how="left", on="id")
)
y_train = train["PitNextLap"]

cv = pd.read_parquet(PROJECT_ROOT / "data" / "cv.parquet")

logreg = LogisticRegression(max_iter=5000, solver="lbfgs", C=0.1)

# **OOF PREDICTIONS**

In [2]:
scores = []
oof = pd.Series(index=train.index, dtype=float, name="oof")

for k in sorted(cv.outer_fold.unique()):
    is_val = cv["outer_fold"] == k
    logreg.fit(X_train[~is_val], y_train[~is_val])

    oof[is_val] = logreg.predict_proba(X_train[is_val])[:, 1]
    scores.append(roc_auc_score(y_train[is_val], oof[is_val]))

print(scores)
print("OOF ROC-AUC:", roc_auc_score(y_train, oof))

[0.9185086795223968, 0.9185257253572292, 0.9170512836902583, 0.9168607598399791, 0.9172703359757756]
OOF ROC-AUC: 0.9176398379097198


# **TEST PREDICTIONS**

In [3]:
logreg.fit(X_train, y_train)
preds = pd.Series(logreg.predict_proba(X_test)[:, 1], index=test.index, dtype=float, name="preds")

# **EXPORT**

In [7]:
oof.to_frame().to_parquet(PROJECT_ROOT / "data" / "oof" / "003-oof.parquet")
preds.to_frame().to_parquet(PROJECT_ROOT / "data" / "preds" / "003-preds.parquet")